In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [7]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI in the format 's3://bucket-name/key'. The function should return the region by making a boto3 call to get bucket location."}, {'task': "Create a JSON policy document that allows an IAM role to read and list objects from a specific S3 bucket named 'my-data-bucket' and all its contents."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which follow the pattern of 'i-' followed by exactly 17 hexadecimal characters."}]


In [8]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [10]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [11]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a complete solution with explanations and examples:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\nfrom typing import Optional\n\n\ndef extract_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    \"\"\"\n    Extract the AWS region from an S3 bucket URI by calling get_bucket_location.\n    \n    Args:\n        s3_uri (str): S3 URI in the format 's3://bucket-name/key' or 's3://bucket-name'\n        \n    Returns:\n        Optional[str]: The AWS region name, or None if the bucket is in us-east-1\n                      (which returns None from get_bucket_location)\n    \n    Raises:\n        ValueError: If the URI format is invalid\n        ClientError: If there are AWS permission or access issues\n    \"\"\"\n    # Parse the S3 URI\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}. Expected 's3://bucket-name/key'\")\n    \n    # Remove 's3://' 